In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')

DRIVE_FOLDER = '/content/drive/MyDrive/'
if not os.path.exists(DRIVE_FOLDER):
    os.makedirs(DRIVE_FOLDER)
    print(f"Created folder: {DRIVE_FOLDER}. Please drop your 'latency_results.json' file there.")
else:
    print(f"Drive mounted. Target folder exists: {DRIVE_FOLDER}")

In [ ]:
import json
import pandas as pd

json_path = os.path.join(DRIVE_FOLDER, 'latency_results.json')

with open(json_path, 'r') as f:
    data = json.load(f)

rows = []
for query_id, query_data in data.items():
    flat_row = {
        'query_id': query_id,
        'category': query_data.get('category', 'unknown')
    }
    flat_row.update(query_data['latency'])
    rows.append(flat_row)

df = pd.DataFrame(rows)

# Calculate total execution times for each RAG pipeline
df['total_raw'] = df['raw_retrieval'] + df['raw_generation']
df['total_enriched'] = df['query_enrichment'] + df['enriched_retrieval'] + df['enriched_generation']
df['total_reranked'] = df['query_enrichment'] + df['enriched_retrieval'] + df['reranking'] + df['reranked_generation']

print(f"Successfully loaded {len(df)} queries!")
df.head(3)

In [ ]:
import numpy as np
import scipy.stats as stats

metrics = [
    'query_enrichment', 'raw_retrieval', 'raw_generation',
    'enriched_retrieval', 'enriched_generation', 'reranking', 'reranked_generation',
    'total_raw', 'total_enriched', 'total_reranked'
]

summary_stats = []
n = len(df)

for metric in metrics:
    mean_val = df[metric].mean()
    median_val = df[metric].median()
    std_val = df[metric].std()

    # 95% Confidence Interval using standard error of the mean
    sem = stats.sem(df[metric])
    ci_lower, ci_upper = stats.t.interval(0.95, df=n-1, loc=mean_val, scale=sem)

    summary_stats.append({
        'Metric Stage': metric,
        'Mean (s)': mean_val,
        'Median (s)': median_val,
        'Std Dev (s)': std_val,
        '95% CI Lower': ci_lower,
        '95% CI Upper': ci_upper
    })

summary_df = pd.DataFrame(summary_stats)

summary_df.to_csv(os.path.join(DRIVE_FOLDER, 'latency_summary_statistics.csv'), index=False)
print("Summary statistics calculated and saved to Drive!")
summary_df

In [ ]:
# Total end-to-end architectures
pipelines = summary_df[summary_df['Metric Stage'].str.startswith('total_')].copy()
pipelines['Label'] = ['Raw RAG', 'Enriched RAG', 'Reranked RAG']

pipelines['Error'] = pipelines['95% CI Upper'] - pipelines['Mean (s)']

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#880E4F', '#C2185B', '#F8BBD0']

bars = ax.bar(
    pipelines['Label'], pipelines['Mean (s)'],
    yerr=pipelines['Error'], capsize=6,
    color=colors, edgecolor='black', alpha=0.85
)

ax.set_ylabel('Total Latency (seconds)', fontsize=12)
ax.set_title('Total Architecture Latency Comparison (with 95% CI)', fontsize=14, fontweight='bold', pad=15)
ax.grid(axis='y', linestyle='--', alpha=0.5)

for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2.0, yval + 0.5, f'{yval:.2f}s', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(DRIVE_FOLDER, 'pipeline_latency_comparison.png'), dpi=300)
plt.show()

In [ ]:
# Group by intent category and pull structural pipeline means
category_data = df.groupby('category')[['total_raw', 'total_enriched', 'total_reranked']].mean()

fig, ax = plt.subplots(figsize=(10, 6))
category_data.plot(kind='bar', ax=ax, color=['#880E4F', '#C2185B', '#F8BBD0'], edgecolor='black', alpha=0.85)

ax.set_ylabel('Mean Latency (seconds)', fontsize=12)
ax.set_xlabel('User Query Category', fontsize=12)
ax.set_title('Mean Pipeline Latency Segmented by Query Category', fontsize=14, fontweight='bold', pad=15)
ax.grid(axis='y', linestyle='--', alpha=0.5)
ax.legend(['Raw RAG Pipeline', 'Enriched RAG Pipeline', 'Reranked RAG Pipeline'], title='Architecture Strategy')
plt.xticks(rotation=0)

plt.tight_layout()
plt.savefig(os.path.join(DRIVE_FOLDER, 'category_latency_comparison.png'), dpi=300)
plt.show()

In [ ]:
# Run paired t-tests
t_raw_vs_enriched = stats.ttest_rel(df['total_raw'], df['total_enriched'])
t_enriched_vs_reranked = stats.ttest_rel(df['total_enriched'], df['total_reranked'])

print("--- Hypothesis Testing Results ---")
print(f"Raw RAG vs. Enriched RAG p-value: {t_raw_vs_enriched.pvalue:.4f}")
print(f"Enriched RAG vs. Reranked RAG p-value: {t_enriched_vs_reranked.pvalue:.4f}")

print("\n--- Conclusion ---")
if t_raw_vs_enriched.pvalue > 0.05:
    print("The latency differences between your architectures are NOT statistically significant (p > 0.05).")
    print("This means adding enrichment or reranking steps does not meaningfully slow down your system;")
    print("the variability is entirely driven by the LLM generation times.")
else:
    print("You have statistically significant variations in pipeline execution speed!")

In [ ]:
import seaborn as sns

df_melted = df.melt(
    id_vars=['query_id'],
    value_vars=['total_raw', 'total_enriched', 'total_reranked'],
    var_name='Pipeline', value_name='Latency (s)'
)

df_melted['Pipeline'] = df_melted['Pipeline'].map({
    'total_raw': 'Raw RAG',
    'total_enriched': 'Enriched RAG',
    'total_reranked': 'Reranked RAG'
})

fig, ax = plt.subplots(figsize=(8, 5))

pink_palette = ['#880E4F', '#C2185B', '#F8BBD0']

sns.boxplot(x='Pipeline', y='Latency (s)', data=df_melted, palette=pink_palette, width=0.5, ax=ax)
sns.stripplot(x='Pipeline', y='Latency (s)', data=df_melted, color='#4A0E17', alpha=0.3, jitter=0.15, size=4)

ax.set_ylabel('Total Execution Time (seconds)', fontsize=11)
ax.set_xlabel('System Configuration Architecture', fontsize=11)
ax.set_title('Distribution of Total Latency Across RAG Iterations', fontsize=13, fontweight='bold', pad=15, color='#880E4F')
ax.grid(axis='y', linestyle=':', alpha=0.6)

plt.tight_layout()
plt.savefig(os.path.join(DRIVE_FOLDER, 'latency_distribution_boxplot.png'), dpi=300)
plt.show()